In [ ]:
#install torch

In [ ]:
!pip install transformers request beautifulsoup pandas numpy

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import requests
from bs4 import BeautifulSoup
import re

Instantiate Model

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("nlptown/bert-base-multilingual-uncased-sentiment")
model = AutoModelForSequenceClassification.from_pretrained("nlptown/bert-base-multilingual-uncased-sentiment")

Encode and Calculate Sentiment

In [ ]:
tokens = tokenizer.encode("I am so happy!", return_tensor = "pt")

In [ ]:
tokenizer.decode(tokens[0])

In [ ]:
result = model(tokens)

In [ ]:
result.logits

In [ ]:
int(torch.argmax(result.logits)) = 1

Review collection

In [ ]:
r = requests.get()
soup = BeautifulSoup(r.text, "html.parser")
regex = re.complex('."comment."')
results = soup.find.all('p',{'class':regex})
reviews = [result.text for result in results]


In [ ]:
results

Score

In [ ]:
df = pd.Dataframe(np.array(reviews), culumns=['review'])

In [ ]:
df.tail()

In [ ]:
def Sentiment_score(reviews):
    tokens = tokenizer(reviews, return_tensors='pt')
    result = model(tokens)
    return int(torch.argmax(result.logits))=1

In [ ]:
Sentiment_score(df['review'].iloc[1])

In [ ]:
df['sentiment'] = df['review'].apply(lambda x: Sentiment_score(x[:512]))

In [ ]:
df['review']

Finetunning the model

In [ ]:
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score
from transformers import Trainer, TrainingArguments
from transformers import BertTokenizer, BertForSequenceClassification
import torch

In [ ]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertForSequenceClassification.from_pretrained('bert-base-uncased')

custome dataset -> torch dataset

In [ ]:
class Dataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        encoding = tokenizer(text, truncation=True, padding='max_length', max_length=512, return_tensors='pt')
        return {**encoding, 'labels': torch.tensor(label)}

In [ ]:
train_datset = Dataset(X_train, y_train)
train_datset = Dataset(X_test, y_test)

Evaluation Matrics

In [ ]:
def compute_metrics(p):
    pred, labels = p
    pred = np.argmax(pred, axis=1)

    accuracy = accuracy_score(labels, pred)
    recall = recall_score(labels, pred, average='weighted')
    precision = precision_score(labels, pred, average='weighted')
    f1 = f1_score(labels, pred, average='weighted')

    return {
        'accuracy': accuracy,
        'recall': recall,
        'precision': precision,
        'f1': f1
    }

In [ ]:
args = TrainingArguments(
    output_dir = "output",
    num_train_epochs = 3,
    per_device_train_batch_size = 8,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_datset,
    eval_dataset=train_datset,
    compute_metrics=compute_metrics
)

In [ ]:
trainer.train()

In [ ]:
trainer.evaluate()

In [ ]:
trainer.save_model("sentiment_model")

Loading custome model

In [ ]:
model_2 = BertForSequenceClassification.from_pretrained("sentiment_model")